In [ ]:
import sqlalchemy
import pandas as pd
import numpy as np
from google.colab import userdata
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as mtick
import seaborn as sns

In [ ]:
DB_USER = userdata.get('DB_USER')
DB_PASSWORD = userdata.get('DB_PASSWORD')
DB_HOST = userdata.get('DB_HOST')
DB_NAME = userdata.get('DB_NAME')
DB_PORT = "5432"

In [ ]:
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = sqlalchemy.create_engine(connection_string)

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
abmf_hoffice_24_30_data = """
SELECT
    timestamp,
    gateway_serial,
    active_power_overall_total,
    apparent_power_overall_total,
    power_factor_overall,
    "total_system_kWh",
    load,
    frequency,
    active_assets,
    active_asset_count,
    workhour,
    transformer_capacity,
    transformer_load_percentage,
    line_to_neutral_voltage_phase_a,
    line_to_neutral_voltage_phase_b,
    line_to_neutral_voltage_phase_c,
    line_current_overall_phase_a,
    line_current_overall_phase_b,
    line_current_overall_phase_c,
    voltage_unbalance_factor,
    current_unbalance_factor,
    total_harmonic_distortion_current_phase_a,
    total_harmonic_distortion_current_phase_b,
    total_harmonic_distortion_current_phase_c
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515' AND timestamp BETWEEN '2026-05-24 00:00:00' AND '2026-05-30 23:59:59'
ORDER BY timestamp DESC;
"""

abmf_hoffice_24_30_df = pd.read_sql(abmf_hoffice_24_30_data, engine)

In [ ]:
abmf_hoffice_24_30_df[(abmf_hoffice_24_30_df['active_assets'] == 'Grid') & ((abmf_hoffice_24_30_df['line_current_overall_phase_a'] + abmf_hoffice_24_30_df['line_current_overall_phase_b'] + abmf_hoffice_24_30_df['line_current_overall_phase_c']) != 0)][['line_current_overall_phase_a', 'line_current_overall_phase_b', 'line_current_overall_phase_c']]

In [ ]:
abmf_hoffice_24_30_df[(abmf_hoffice_24_30_df['active_assets'] == 'Grid') & ((abmf_hoffice_24_30_df['line_current_overall_phase_a'] + abmf_hoffice_24_30_df['line_current_overall_phase_b'] + abmf_hoffice_24_30_df['line_current_overall_phase_c']) != 0) & (abmf_hoffice_24_30_df['line_current_overall_phase_a'] == 0) & (abmf_hoffice_24_30_df['line_current_overall_phase_b'] == 0)][['line_current_overall_phase_a', 'line_current_overall_phase_b', 'line_current_overall_phase_c']]

In [ ]:
abmf_hoffice_24_30_df[(abmf_hoffice_24_30_df['active_assets'] == 'Grid') & ((abmf_hoffice_24_30_df['line_current_overall_phase_a'] + abmf_hoffice_24_30_df['line_current_overall_phase_b'] + abmf_hoffice_24_30_df['line_current_overall_phase_c']) != 0) & (abmf_hoffice_24_30_df['line_current_overall_phase_b'] == 0) & (abmf_hoffice_24_30_df['line_current_overall_phase_c'] == 0)][['line_current_overall_phase_a', 'line_current_overall_phase_b', 'line_current_overall_phase_c']]

In [ ]:
(abmf_hoffice_24_30_df[(abmf_hoffice_24_30_df['active_assets'] == 'Grid') & ((abmf_hoffice_24_30_df['line_current_overall_phase_a'] + abmf_hoffice_24_30_df['line_current_overall_phase_b'] + abmf_hoffice_24_30_df['line_current_overall_phase_c']) != 0)][['line_current_overall_phase_a', 'line_current_overall_phase_b', 'line_current_overall_phase_c']]).mean(axis=0)

In [ ]:
(abmf_hoffice_24_30_df[(abmf_hoffice_24_30_df['active_assets'] == 'Grid') & ((abmf_hoffice_24_30_df['line_current_overall_phase_a'] + abmf_hoffice_24_30_df['line_current_overall_phase_b'] + abmf_hoffice_24_30_df['line_current_overall_phase_c']) != 0)][['line_current_overall_phase_a', 'line_current_overall_phase_b', 'line_current_overall_phase_c']]).min(axis=0)

In [ ]:
(abmf_hoffice_24_30_df[(abmf_hoffice_24_30_df['active_assets'] == 'Grid') & ((abmf_hoffice_24_30_df['line_current_overall_phase_a'] + abmf_hoffice_24_30_df['line_current_overall_phase_b'] + abmf_hoffice_24_30_df['line_current_overall_phase_c']) != 0)][['line_current_overall_phase_a', 'line_current_overall_phase_b', 'line_current_overall_phase_c']]).max(axis=0)

1. Total Energy ✅
2. Load Heatmap ✅
3. Three-phase voltage
4. Frequency ✅
5. Generator Load Utilization ✅
6. Average Power Factor
7. Peak Demand ✅
8. Source Switches (?)

In [ ]:
conditions = [
    abmf_hoffice_24_30_df['active_assets'] == 'Grid',
    abmf_hoffice_24_30_df['active_assets'] == 'Generator 1',
    abmf_hoffice_24_30_df['active_assets'] == ''
]

choices = [200, 88, 0]

abmf_hoffice_24_30_df['asset_capacity'] = np.select(conditions, choices, default=0)

In [ ]:
abmf_hoffice_24_30_df['power_factor_overall'] = abs(abmf_hoffice_24_30_df['active_power_overall_total'] / abmf_hoffice_24_30_df['apparent_power_overall_total'])
abmf_hoffice_24_30_df['asset_load_percentage'] = abs(abmf_hoffice_24_30_df['apparent_power_overall_total'] * 100 / abmf_hoffice_24_30_df['asset_capacity'])

In [ ]:
abmf_hoffice_24_30_df['average_total_harmonic_distortion'] = abmf_hoffice_24_30_df[['total_harmonic_distortion_current_phase_a','total_harmonic_distortion_current_phase_b', 'total_harmonic_distortion_current_phase_c']].mean(axis=1)

In [ ]:
df = abmf_hoffice_24_30_df.copy()

In [ ]:
abmf_hoffice_24_30_df_copy = abmf_hoffice_24_30_df.copy()

In [ ]:
# 1. Convert the column to datetime (just in case it loaded as a string/object)
abmf_hoffice_24_30_df_copy["timestamp"] = pd.to_datetime(abmf_hoffice_24_30_df_copy["timestamp"])

# 2. Set the column as the index
abmf_hoffice_24_30_df_copy = abmf_hoffice_24_30_df_copy.set_index("timestamp")

# 3. Crucial: Sort the index chronologically
abmf_hoffice_24_30_df_copy = abmf_hoffice_24_30_df_copy.sort_index()

In [ ]:
df_filled = abmf_hoffice_24_30_df_copy.resample("1min").first()

In [ ]:
df_filled.isna().sum()

In [ ]:
# turn 'timestamp' into a regular column
df = df_filled.reset_index()

# Rename index into timestamp if not currently nameed timestamp
if "timestamp" not in df.columns and "index" in df.columns:
    df = df.rename(columns={"index": "timestamp"})

# Clean, sort, and parse data
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Replace empty strings and NaNs with clear tracking labels
df["active_assets"] = df["active_assets"].replace(
    {"": "None / Standby", np.nan: "No Power"}
)

# Fill NaN load values with 0 to show a visible line during outages
df["load"] = df["load"].fillna(0)

In [ ]:
df.isna().sum()

In [ ]:
df[df['load'] != df['active_power_overall_total']].active_assets.value_counts()

Some active_power_overall_total values were negative. That's why the sum was different from the load sum

Seems logic for total_systm_kWh is wrong. Confirm

Seems it's correct.

In [ ]:
# 1. Filter out 'No Power' and 'None / Standby' before grouping
df_filtered = df[~df["active_assets"].isin(["No Power", "None / Standby"])]

# 2. Group by active_assets and sum total_system_kWh
grouped_df = (
    df_filtered.groupby("active_assets")["total_system_kWh"].sum().reset_index()
)

# 3. Convert to percentages based on the remaining active assets
total_kWh = grouped_df["total_system_kWh"].sum()
grouped_df["percentage"] = (grouped_df["total_system_kWh"] / total_kWh) * 100

# 4. Sort the values in descending order
grouped_df = grouped_df.sort_values(by="percentage", ascending=False)

# 5. Define the custom color scheme
custom_palette = {
    "Grid": "#1f77b4",  # Blue
    "Generator 1": "#ff7f0e",  # Orange
}

colors = [
    custom_palette.get(asset, "#7f7f7f") for asset in grouped_df["active_assets"]
]

# 6. Generate the plot
fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(
    grouped_df["active_assets"],
    grouped_df["percentage"],
    color=colors,
    edgecolor="black",
    width=0.6,
)

# Add percentage text on top of each bar
ax.bar_label(bars, fmt="%.1f%%", padding=5, fontsize=11)

# Styling and formatting adjustments
ax.set_title(
    "Total System Consumption by Active Assets (%)",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
ax.set_xlabel("Active Assets", fontsize=11, labelpad=10)
ax.set_ylabel("Percentage of Total Consumption", fontsize=11, labelpad=10)

# --- ADDED CODE TO REMOVE BOX LINES ---
# Remove the top and right borders (spines)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# --------------------------------------

# Explicitly remove grid lines completely
ax.grid(False)

# Format y-axis as percentage
ax.yaxis.set_major_formatter(mtick.PercentFormatter())

# Expand y-limit slightly to ensure labels aren't cut off at the top
ax.set_ylim(0, max(grouped_df["percentage"]) * 1.15)

# Keep labels straight since there are fewer categories now
plt.xticks(rotation=0)

# Save the visualization
plt.savefig("total_system_percentage_by_asset.png", bbox_inches="tight")

In [ ]:
# 1. Filter out 'No Power' and 'None / Standby' before grouping
df_filtered = df[~df["active_assets"].isin(["No Power", "None / Standby"])]

# 2. Group by active_assets and sum total_system_kWh
grouped_df = (
    df_filtered.groupby("active_assets")["total_system_kWh"].sum().reset_index()
)

# 3. Convert to percentages based on the remaining active assets
total_kWh = grouped_df["total_system_kWh"].sum()
grouped_df["percentage"] = (grouped_df["total_system_kWh"] / total_kWh) * 100

# 4. Sort the values in descending order
grouped_df = grouped_df.sort_values(by="percentage", ascending=False)

# 5. Define the custom color scheme
custom_palette = {
    "Grid": "#1f77b4",  # Blue
    "Generator 1": "#ff7f0e",  # Orange
}

colors = [
    custom_palette.get(asset, "#7f7f7f") for asset in grouped_df["active_assets"]
]

# 6. Generate the Doughnut Chart
fig, ax = plt.subplots(figsize=(6, 5))

wedges, texts, autotexts = ax.pie(
    grouped_df["percentage"],
    labels=grouped_df["active_assets"],
    autopct="%.1f%%",
    startangle=90,
    colors=colors,
    # KEY DIFFERENCE: width=0.4 punches out the center hole
    wedgeprops=dict(edgecolor="black", linewidth=0.8, width=0.4),
    textprops=dict(fontsize=11),
    pctdistance=0.8  # Pushes the text outward to center it in the ring
)

# Style the percentage text inside the ring for high contrast
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_weight('bold')

# Styling and formatting adjustments
ax.set_title(
    "Total System Consumption by Active Assets (%)",
    fontsize=13,
    fontweight="bold",
    pad=15,
)

plt.tight_layout()
plt.savefig("total_system_doughnut_chart.png", bbox_inches="tight")

In [ ]:
grouped_df

In [ ]:
df.active_assets.value_counts(normalize=True)

In [ ]:
counts = df['active_assets'].value_counts()
percentages = (counts / counts.sum() * 100).round(0).astype(int)

In [ ]:
percentages

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

custom_palette = {
    "Grid": "#1f77b4",
    "Generator 1": "#ff7f0e",
    "None / Standby": "#7f7f7f",
    "No Power": "#000000",
}

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    x=percentages.index,
    y=percentages.values,
    palette=custom_palette,
    hue=percentages.index,
    legend=False
)

ax.grid(False)

# --- ADDED CODE TO REMOVE BOX LINES ---
# Remove the top and right borders (spines)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# --------------------------------------

for container in ax.containers:
    ax.bar_label(container, fmt='%d%%', padding=3)

plt.title("Power Source Distribution", fontsize=14, pad=15)
plt.xlabel("Source Type", fontsize=12)
plt.ylabel("Percentage (%)", fontsize=12)
plt.ylim(0, 110)

plt.tight_layout()
plt.savefig('power_source_distribution.png')
plt.show()


In [ ]:
df.groupby('active_assets').load.mean()

In [ ]:
abmf_hoffice_24_30_df.groupby('active_assets').load.mean()

In [ ]:
abmf_hoffice_24_30_df.groupby('active_assets').load.min()

In [ ]:
df.groupby('active_assets').load.max()

In [ ]:
# 1. Target your specific generator and grid assets (adjust exact string names if needed)
assets_to_plot = ["Grid", "Generator 1"]
filtered_df = abmf_hoffice_24_30_df[
    abmf_hoffice_24_30_df["active_assets"].isin(assets_to_plot)
]

# 2. Group by active_assets and calculate min, mean (average), and max simultaneously
summary_df = (
    filtered_df.groupby("active_assets")["load"]
    .agg(["min", "mean", "max"])
    .reset_index()
)

# 3. Sort your categories by their mean load descending
summary_df = summary_df.sort_values(by="mean", ascending=False)

# 4. Initialize the grouped bar chart canvas
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(summary_df["active_assets"]))
width = 0.22  # Optimal thickness for side-by-side bars

# Shifting from "status/risk" to "magnitude"
rects_min = ax.bar(x - width, summary_df['min'], width, label='Minimum', color='#a1c4fd')  # Light Blue
rects_avg = ax.bar(x, summary_df['mean'], width, label='Average', color='#4a90e2')       # Medium Blue
rects_max = ax.bar(x + width, summary_df['max'], width, label='Maximum', color='#003366')  # Deep Navy Blue

# 6. Styling, titles, and text label formatting
ax.set_title(
    "Load Analysis: Min, Avg, and Max Load by Asset",
    fontsize=14,
    fontweight="bold",
    pad=15,
)
ax.set_xlabel("Active Assets", fontsize=12, labelpad=10)
ax.set_ylabel("Load (kW)", fontsize=12, labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels(summary_df["active_assets"], fontsize=11)
ax.legend(fontsize=11)

# Keep the background clean with no grid lines
ax.grid(False)

# --- ADDED CODE TO REMOVE BOX LINES ---
# Remove the top and right borders (spines)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# --------------------------------------

# Add precise value labels on top of every single bar
ax.bar_label(rects_min, padding=3, fmt="%.0f kW", fontsize=9)
ax.bar_label(rects_avg, padding=3, fmt="%.0f kW", fontsize=9, fontweight="bold")
ax.bar_label(rects_max, padding=3, fmt="%.0f kW", fontsize=9)

# Expand the y-axis ceiling slightly so the text labels don't get squished
ax.set_ylim(0, max(summary_df["max"]) * 1.15)

# Save visualization layout cleanly
plt.tight_layout()
plt.savefig("generator_grid_load_summary.png", bbox_inches="tight")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Convert timestamp column to datetime and extract features
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["day_of_week"] = df["timestamp"].dt.day_name()
df["hour_of_day"] = df["timestamp"].dt.hour

# Re-arranged list to start strictly from Sunday
days_order = [
    "Sunday",
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
]
df["day_of_week"] = pd.Categorical(
    df["day_of_week"], categories=days_order, ordered=True
)

# Filter for core assets
df_filtered = df[df["active_assets"].isin(["Grid", "Generator 1"])]

v_min = 0
v_max = df_filtered["load"].max()

# 2. Pivot data and fill inactive/missing intervals with 0
grid_pivot = (
    df_filtered[df_filtered["active_assets"] == "Grid"]
    .pivot_table(
        index="day_of_week",
        columns="hour_of_day",
        values="load",
        aggfunc="mean",
        fill_value=0,
        observed=False,
    )
    .reindex(index=days_order, columns=range(24), fill_value=0)
)

gen_pivot = (
    df_filtered[df_filtered["active_assets"] == "Generator 1"]
    .pivot_table(
        index="day_of_week",
        columns="hour_of_day",
        values="load",
        aggfunc="mean",
        fill_value=0,
        observed=False,
    )
    .reindex(index=days_order, columns=range(24), fill_value=0)
)

# 3. Construct side-by-side layout
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)

# Plot Left Panel: Grid Heatmap
sns.heatmap(
    grid_pivot,
    ax=ax1,
    cmap="Blues",
    vmin=v_min,
    vmax=v_max,
    cbar=True,
    linewidths=0.5,
    linecolor="#f2f2f2",
)
ax1.set_title("Grid Load Profile (kW)", fontsize=13, pad=10)
ax1.set_ylabel("Day of Week", fontsize=11, labelpad=8)
ax1.set_xlabel("")  # FIXED: Explicitly blank out Seaborn's auto-generated label
ax1.grid(False)

# Plot Right Panel: Generator Heatmap
sns.heatmap(
    gen_pivot,
    ax=ax2,
    cmap="Oranges",
    vmin=v_min,
    vmax=v_max,
    cbar=True,
    linewidths=0.5,
    linecolor="#f2f2f2",
)
ax2.set_title("Generator 1 Load Profile (kW)", fontsize=13, pad=10)
ax2.set_ylabel("", labelpad=0)
ax2.set_xlabel("")  # FIXED: Explicitly blank out Seaborn's auto-generated label
ax2.grid(False)

# --- GLOBAL LABELS ---
# Main title adjustments
#fig.suptitle(
#    "Weekly Load Heatmap Comparison", fontsize=15, fontweight="bold", y=0.98
#)

# Shared Figure X-Axis Label (Centered across both plots)
fig.supxlabel("Hour of Day", fontsize=11, y=0.02)
# ----------------------------

# Format horizontal tick parameters
ax1.tick_params(axis="x", rotation=0)
ax2.tick_params(axis="x", rotation=0)

# Save the final visualization layout
plt.savefig("weekly_load_heatmap_sunday_start.png", bbox_inches="tight")
plt.show()

In [ ]:
# 1. Standardize timestamp structure
df = df_filled.reset_index()

if "timestamp" not in df.columns and "index" in df.columns:
    df = df.rename(columns={"index": "timestamp"})

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# 2. Handle asset labels
df["active_assets"] = df["active_assets"].replace(
    {"": "None / Standby", np.nan: "No Power"}
)

# Calculate the unique segment ID before filtering to cleanly break line continuities
df["segment_id"] = (df["active_assets"] != df["active_assets"].shift()).cumsum()

# 3. CHANGE: Filter to include Grid, Generator 1, AND None / Standby
# This intentionally leaves out "No Power" to create physical gaps during outages
assets_to_plot = ["Grid", "Generator 1", "None / Standby"]
df_plot = df[df["active_assets"].isin(assets_to_plot)].copy()

# 4. Calculate realistic Y-limits based on the plotted states
raw_min = df_plot["frequency"].min()
raw_max = df_plot["frequency"].max()
y_span = raw_max - raw_min

# 5% safety margin around operational data bounds
global_y_min = raw_min - (y_span * 0.05)
global_y_max = raw_max + (y_span * 0.05)

# CHANGE: Restored "None / Standby" to the palette and order configurations
hue_order = ["Grid", "Generator 1", "None / Standby"]
custom_palette = {
    "Grid": "#1f77b4",  # Blue
    "Generator 1": "#ff7f0e",  # Orange
    "None / Standby": "#7f7f7f",  # Gray
}

# 5. Generate the Full Week Frequency Profile Plot
start_date = df_plot["timestamp"].min().strftime("%Y-%m-%d")
end_date = df_plot["timestamp"].max().strftime("%Y-%m-%d")

fig, ax = plt.subplots(figsize=(15, 6))

sns.lineplot(
    data=df_plot,
    x="timestamp",
    y="frequency",
    hue="active_assets",
    hue_order=hue_order,
    units="segment_id",
    estimator=None,
    palette=custom_palette,
    linewidth=1.5,
    alpha=0.9,
    ax=ax,
)

# Format horizontal timeline for daily tick marks
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a, %b %d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.set_xlim(df_plot["timestamp"].min(), df_plot["timestamp"].max())

# Apply the realistic frequency boundaries
ax.set_ylim(global_y_min, global_y_max)

# Descriptive labels and text formatting
ax.set_title(
    f"ABMF System Frequency Profile — {start_date} till {end_date}",
    fontsize=14,
    pad=15,
    weight="bold",
)
ax.set_xlabel("Day of Week", fontsize=12, labelpad=10)
ax.set_ylabel("Frequency (Hz)", fontsize=12, labelpad=10)
ax.grid(True, linestyle=":", alpha=0.5)

# --- ADDED CODE TO REMOVE BOX LINES ---
# Remove the top and right borders (spines)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# --------------------------------------

# Clean, deduplicated legend layout
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(
    [by_label[lbl] for lbl in hue_order if lbl in by_label],
    [lbl for lbl in hue_order if lbl in by_label],
    title="System State",
    loc="upper right",
)

plt.tight_layout()

# Save the plot directly
weekly_freq_filename = f"ABMF_Frequency_Profile_{start_date}_to_{end_date}.png"
plt.savefig(weekly_freq_filename, dpi=300)
print(
    f"Weekly frequency plot (with Standby and gaps) saved as: '{weekly_freq_filename}'"
)

In [ ]:
# --- CONFIGURATION & CONSTANTS ---
GENSET_CAPACITY_KW = 88  # Change this to your generator's actual size in kW
GENSET_EFF_LOW = 40  # Lower boundary of high-efficiency zone (%)
GENSET_EFF_HIGH = 80  # Upper boundary of high-efficiency zone (%)

# Colors matching your clean aesthetic
C_EFF = "#2ca02c"  # Green for the sweet-spot efficiency zone
C_NEUTRAL = "#b0bec5"  # Muted slate gray for under/over-utilized zones

# 1. Filter for Generator 1 data and drop missing records
gen_df = df[df["active_assets"] == "Generator 1"].dropna(subset=["load"])

# 2. Convert absolute kW load into a percentage of total capacity
# (If your 'load' column is already a 0-100% value, you can skip this calculation)
gen_load_pct = (gen_df["load"] / GENSET_CAPACITY_KW) * 100

# 3. Calculate the histogram distributions (0% to 100% in buckets of 10%)
bins = np.arange(0, 101, 10)
counts, edges = np.histogram(gen_load_pct, bins=bins)
centers = (edges[:-1] + edges[1:]) / 2

# 4. CONVERT MINUTES TO HOURS:
# Since data is logged every minute, divide the raw row counts by 60
counts_hours = counts / 60.0

# 5. Dynamically assign colors based on the efficiency sweet-spot
colors = [
    C_EFF if GENSET_EFF_LOW <= c < GENSET_EFF_HIGH else C_NEUTRAL
    for c in centers
]

# 6. Generate the sleek, low-height utilization plot
fig, ax = plt.subplots(figsize=(8, 3.5))

# Plot utilization bars
bars = ax.bar(centers, counts_hours, width=8, color=colors, edgecolor="black")

# Highlight the optimal efficiency window in the background
ax.axvspan(GENSET_EFF_LOW, GENSET_EFF_HIGH, color=C_EFF, alpha=0.06)

# Labels, titles, and layout formatting
ax.set_title("Generator Load Utilization Profile", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Generator Load (%)", fontsize=11, labelpad=8)
ax.set_ylabel("Operation Time (Hours)", fontsize=11, labelpad=8)

# Clean rendering adjustments
ax.set_xlim(0, 100)
ax.set_xticks(bins)

# --- ADDED CODE TO REMOVE BOX LINES ---
# Remove the top and right borders (spines)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# --------------------------------------

ax.grid(False)  # Completely removes background gridlines per preference

# Optional: Add text label inside the efficiency zone
ax.text(
    (GENSET_EFF_LOW + GENSET_EFF_HIGH) / 2,
    max(counts_hours) * 0.9,
    "Optimal Efficiency Zone",
    color=C_EFF,
    fontsize=10,
    fontweight="bold",
    ha="center",
)

plt.tight_layout()
plt.savefig("generator_load_utilization.png", dpi=300)

In [ ]:
# 1. Clean data: Filter out impossible sensor anomalies
valid_data = abmf_hoffice_24_30_df[
    (abmf_hoffice_24_30_df["apparent_power_overall_total"] > 0)
    & (
        abmf_hoffice_24_30_df["load"]
        <= abmf_hoffice_24_30_df["apparent_power_overall_total"]
    )
].copy()

# 2. Filter exclusively for Grid and Generator assets
target_assets = ["Grid", "Generator 1"]
valid_data = valid_data[valid_data["active_assets"].isin(target_assets)]

# 3. Calculate instantaneous power factor per minute for MIN and MAX extraction
valid_data["pf_instantaneous"] = (
    valid_data["load"] / valid_data["apparent_power_overall_total"]
)

# 4. Extract Min and Max boundaries
min_max_df = (
    valid_data.groupby("active_assets")["pf_instantaneous"]
    .agg(["min", "max"])
    .reset_index()
)

# 5. Extract sums for the true Energy-Weighted Average
sums_df = (
    valid_data.groupby("active_assets")[
        ["load", "apparent_power_overall_total"]
    ]
    .sum()
    .reset_index()
)

# 6. Merge metrics and calculate the final weighted mean
summary_df = pd.merge(min_max_df, sums_df, on="active_assets")
summary_df["mean"] = (
    summary_df["load"] / summary_df["apparent_power_overall_total"]
)

# Sort by average performance for visualization layout
summary_df = summary_df.sort_values(by="mean", ascending=False)

# 7. Construct Grouped Bar Chart
fig, ax = plt.subplots(figsize=(10, 5.5))

x = np.arange(len(summary_df["active_assets"]))
width = 0.22  # Balanced spacing for 3 metrics side-by-side

# Neutral, modern color palette (avoids alert red)
c_min = "#d32f2f"  # Light silver-gray
c_avg = "#4a90e2"  # Slate corporate blue
c_max = "#1c3d5a"  # Deep dark navy

# Plot Min, Avg, Max bars
rects_min = ax.bar(
    x - width,
    summary_df["min"],
    width,
    label="Minimum",
    color=c_min,
    edgecolor="black",
)
rects_avg = ax.bar(
    x,
    summary_df["mean"],
    width,
    label="Weighted Average",
    color=c_avg,
    edgecolor="black",
)
rects_max = ax.bar(
    x + width,
    summary_df["max"],
    width,
    label="Maximum",
    color=c_max,
    edgecolor="black",
)

# Styling and Labels
ax.set_title(
    "Power Factor Analysis: Min, Weighted Avg, and Max by Asset",
    fontsize=14,
    fontweight="bold",
    pad=15,
)
ax.set_xlabel("Active Assets", fontsize=12, labelpad=10)
ax.set_ylabel("Power Factor", fontsize=12, labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels(summary_df["active_assets"], fontsize=11)
ax.set_ylim(0, 1.18)  # Room for labels without ceiling truncation
ax.legend(fontsize=11, loc="upper left")

# --- ADDED CODE TO REMOVE BOX LINES ---
# Remove the top and right borders (spines)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# --------------------------------------

# Remove background gridlines entirely
ax.grid(False)

# Add clear 2-decimal values on top of every bar
ax.bar_label(rects_min, padding=4, fmt="%.2f", fontsize=9.5)
ax.bar_label(rects_avg, padding=4, fmt="%.2f", fontsize=9.5, fontweight="bold")
ax.bar_label(rects_max, padding=4, fmt="%.2f", fontsize=9.5)

plt.tight_layout()
plt.savefig("power_factor_summary_comparison.png", dpi=300)

In [ ]:
# 1. Clean data: Filter out impossible sensor anomalies
valid_data = abmf_hoffice_24_30_df[
    (abmf_hoffice_24_30_df["apparent_power_overall_total"] > 0)
    & (
        abmf_hoffice_24_30_df["load"]
        <= abmf_hoffice_24_30_df["apparent_power_overall_total"]
    )
].copy()

# 2. Calculate the instantaneous power factor per minute
valid_data["pf"] = (
    valid_data["load"] / valid_data["apparent_power_overall_total"]
)

# 3. Setup bins focusing on the typical operational window (0.50 to 1.00)
# Steps of 0.02 provide excellent granularity for power factor tracking
bins = np.arange(0, 1.01, 0.02)
centers = (bins[:-1] + bins[1:]) / 2

# Extract data splits
grid_pf = valid_data[valid_data["active_assets"] == "Grid"]["pf"].dropna()
gen_pf = (
    valid_data[valid_data["active_assets"] == "Generator 1"]["pf"].dropna()
)

# Compute distributions
grid_counts, _ = np.histogram(grid_pf, bins=bins)
gen_counts, _ = np.histogram(gen_pf, bins=bins)

# Convert minute-log counts to hours
grid_hours = grid_counts / 60.0
gen_hours = gen_counts / 60.0

# 4. Create side-by-side plots sharing the Y-axis (Hours)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)

# Left Panel: Grid Power Factor Distribution
ax1.bar(
    centers,
    grid_hours,
    width=0.016,
    color="#1f77b4",
    edgecolor="black",
    alpha=0.85,
)
ax1.set_title("Grid Power Factor Stability", fontsize=12)
ax1.set_xlabel("Power Factor (kW / kVA)", fontsize=11)
ax1.set_ylabel("Operation Time (Hours)", fontsize=11)
ax1.set_xlim(0, 1.01)
ax1.grid(False)

# --- ADDED CODE TO REMOVE BOX LINES ---
# Remove the top and right borders (spines)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
# --------------------------------------

# Right Panel: Generator Power Factor Distribution
ax2.bar(
    centers,
    gen_hours,
    width=0.016,
    color="#ff7f0e",
    edgecolor="black",
    alpha=0.85,
)
ax2.set_title(
    "Generator 1 Power Factor Stability", fontsize=12
)
ax2.set_xlabel("Power Factor (kW / kVA)", fontsize=11)
ax2.set_xlim(0, 1.01)
ax2.grid(False)

# --- ADDED CODE TO REMOVE BOX LINES ---
# Remove the top and right borders (spines)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
# --------------------------------------

# Main Title
fig.suptitle(
    "Weekly Power Factor Distribution Profile",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)

plt.tight_layout()
plt.savefig("power_factor_histograms_side_by_side.png", bbox_inches="tight")

In [ ]:
GRID_CAPACITY_KW = 200

In [ ]:
abmf_hoffice_24_30_df.average_total_harmonic_distortion.sample(10)

Harmonic distortion seems the same for all the rows.

In [ ]:
abmf_hoffice_24_30_df[['total_harmonic_distortion_current_phase_a',
       'total_harmonic_distortion_current_phase_b',
       'total_harmonic_distortion_current_phase_c',
       'average_total_harmonic_distortion']].value_counts()

THD is the same for all rows

In [ ]:
abmf_hoffice_24_30_df.voltage_unbalance_factor.hist()

Voltage Unbalance seems okay.

In [ ]:
abmf_hoffice_24_30_df.current_unbalance_factor.hist()

In [ ]:
# Define your bins and structure
bins = np.arange(0, 101, 10)
centers = (bins[:-1] + bins[1:]) / 2
width = 3.5  # Narrower width so they fit side-by-side

# Compute histogram metrics for both assets (assuming 'load_pct' column exists)
grid_pct = (
    df[df["active_assets"] == "Grid"]["load"] / GRID_CAPACITY_KW
) * 100
gen_pct = (
    df[df["active_assets"] == "Generator 1"]["load"] / GENSET_CAPACITY_KW
) * 100

grid_counts, _ = np.histogram(grid_pct.dropna(), bins=bins)
gen_counts, _ = np.histogram(gen_pct.dropna(), bins=bins)

# Convert counts to hours
grid_hours = grid_counts / 60.0
gen_hours = gen_counts / 60.0

# Plot on a single canvas
fig, ax = plt.subplots(figsize=(10, 5))

# Offset the x-positions manually to place them side-by-side
ax.bar(
    centers - (width / 2),
    grid_hours,
    width=width,
    label="Grid",
    color="#1f77b4",
    edgecolor="black",
)
ax.bar(
    centers + (width / 2),
    gen_hours,
    width=width,
    label="Generator 1",
    color="#ff7f0e",
    edgecolor="black",
)

# Highlight efficiency zone
ax.axvspan(40, 80, color="#2ca02c", alpha=0.05, label="Gen Efficiency Zone")

ax.set_title(
    "Asset Load Utilization Comparison", fontsize=13, fontweight="bold"
)
ax.set_xlabel("Load Capacity (%)")
ax.set_ylabel("Operation Time (Hours)")
ax.set_xticks(bins)
ax.grid(False)
ax.legend()

plt.savefig("grouped_utilization_histogram.png", bbox_inches="tight")

In [ ]:
abmf_hoffice_24_30_df[abmf_hoffice_24_30_df['power_factor_overall'] > 0.9]

Current unbalance

Check for average voltage

In [ ]:
abmf_hoffice_24_30_df.columns

In [ ]:
abmf_hoffice_24_30_df[['line_to_neutral_voltage_phase_a','line_to_neutral_voltage_phase_b', 'line_to_neutral_voltage_phase_c']].mean(axis=1).hist()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Clean data & calculate average voltage across all three phases
valid_data = abmf_hoffice_24_30_df.copy()

valid_data["avg_voltage"] = valid_data[
    [
        "line_to_neutral_voltage_phase_a",
        "line_to_neutral_voltage_phase_b",
        "line_to_neutral_voltage_phase_c",
    ]
].mean(axis=1)

# Filter out extreme sensor anomalies (e.g., negative readings or NaN values)
valid_data = valid_data[valid_data["avg_voltage"] > 0].dropna(
    subset=["avg_voltage"]
)

# 2. Setup dynamic bins based on the voltage range in your data
v_min = int(np.floor(valid_data["avg_voltage"].min()))
v_max = int(np.ceil(valid_data["avg_voltage"].max()))
bin_step = 2

bins = np.arange(v_min - bin_step, v_max + (bin_step * 2), bin_step)
centers = (bins[:-1] + bins[1:]) / 2
bar_width = bin_step * 0.8

# 3. Extract data splits per asset
grid_v = valid_data[valid_data["active_assets"] == "Grid"]["avg_voltage"]
gen_v = valid_data[valid_data["active_assets"] == "Generator 1"]["avg_voltage"]

# Compute distributions
grid_counts, _ = np.histogram(grid_v, bins=bins)
gen_counts, _ = np.histogram(gen_v, bins=bins)

# Convert minute-log counts to hours
grid_hours = grid_counts / 60.0
gen_hours = gen_counts / 60.0

# 4. Create side-by-side plots sharing the Y-axis (Hours)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

# Left Panel: Grid Voltage Distribution
ax1.bar(
    centers,
    grid_hours,
    width=bar_width,
    color="#1f77b4",
    edgecolor="black",
    alpha=0.85,
)
ax1.set_title("Grid Voltage Stability", fontsize=12)
ax1.set_ylabel("Operation Time (Hours)", fontsize=11)  # Kept on the left edge
# REMOVED: ax1.set_xlabel
ax1.set_xlim(v_min - 4, v_max + 4)
ax1.grid(False)

# Remove the top and right borders (spines)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# Right Panel: Generator Voltage Distribution
ax2.bar(
    centers,
    gen_hours,
    width=bar_width,
    color="#ff7f0e",
    edgecolor="black",
    alpha=0.85,
)
ax2.set_title("Generator 1 Voltage Stability", fontsize=12)
# REMOVED: ax2.set_xlabel
ax2.set_xlim(v_min - 4, v_max + 4)
ax2.grid(False)

# Remove the top and right borders (spines)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# --- ADDED GLOBAL LABELS ---
# Shared Figure Main Title
fig.suptitle(
    "Weekly Voltage Distribution Profile",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

# Shared Figure X-Axis Label (Centered across the whole figure)
fig.supxlabel("Average Line-to-Neutral Voltage (V)", fontsize=11, y=0.02)
# ----------------------------

plt.tight_layout()
plt.savefig("voltage_histograms_side_by_side.png", bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Clean data: Drop missing values and filter out negative anomalies
valid_data = abmf_hoffice_24_30_df.dropna(subset=["current_unbalance_factor"])
valid_data = valid_data[valid_data["current_unbalance_factor"] >= 0].copy()

# 2. Auto-detect scale (Fraction vs. Percentage) & setup appropriate bins
max_val = valid_data["current_unbalance_factor"].max()

bin_step = 5  # 5% granularity
x_label = "Current Unbalance Factor (%)"
x_max = max(10.0, max_val * 1.1)  # Focus window, default to at least 10% unbalance

bins = np.arange(0, x_max + (bin_step * 2), bin_step)
centers = (bins[:-1] + bins[1:]) / 2
bar_width = bin_step * 0.8

# 3. Extract data splits per asset
grid_cuf = valid_data[valid_data["active_assets"] == "Grid"]["current_unbalance_factor"]
gen_cuf = valid_data[valid_data["active_assets"] == "Generator 1"]["current_unbalance_factor"]

# Compute distributions
grid_counts, _ = np.histogram(grid_cuf, bins=bins)
gen_counts, _ = np.histogram(gen_cuf, bins=bins)

# Convert minute-log counts to hours
grid_hours = grid_counts / 60.0
gen_hours = gen_counts / 60.0

# 4. Create side-by-side plots sharing the Y-axis (Hours)
# Bumped figsize height slightly to 4.5 to clear room for the global label
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

# Left Panel: Grid Current Unbalance Distribution
ax1.bar(
    centers,
    grid_hours,
    width=bar_width,
    color="#1f77b4",
    edgecolor="black",
    alpha=0.85,
)
ax1.set_title("Grid Current Unbalance Profile", fontsize=12)
ax1.set_ylabel("Operation Time (Hours)", fontsize=11)  # Kept on the left edge
# REMOVED: ax1.set_xlabel
ax1.set_xlim(0, x_max)
ax1.grid(False)

# Remove the top and right borders (spines)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# Right Panel: Generator Current Unbalance Distribution
ax2.bar(
    centers,
    gen_hours,
    width=bar_width,
    color="#ff7f0e",
    edgecolor="black",
    alpha=0.85,
)
ax2.set_title("Generator 1 Current Unbalance Profile", fontsize=12)
# REMOVED: ax2.set_xlabel
ax2.set_xlim(0, x_max)
ax2.grid(False)

# Remove the top and right borders (spines)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# --- ADDED GLOBAL LABELS ---
# Shared Figure Main Title
fig.suptitle(
    "Weekly Current Unbalance Factor Distribution",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

# Shared Figure X-Axis Label (Centered across both plots)
fig.supxlabel(x_label, fontsize=11, y=0.02)
# ----------------------------

plt.tight_layout()
plt.savefig("current_unbalance_histograms_side_by_side.png", bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Ensure timestamp structure and extract the calendar date
df = abmf_hoffice_24_30_df.reset_index()
if "timestamp" not in df.columns and "index" in df.columns:
    df = df.rename(columns={"index": "timestamp"})

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["date"] = df["timestamp"].dt.date

# 2. Handle asset labels consistently
df["active_assets"] = df["active_assets"].replace(
    {"": "None / Standby", np.nan: "No Power"}
)

# 3. Group by date and asset, then sum the total kWh consumption
daily_energy = (
    df.groupby(["date", "active_assets"])["total_system_kWh"]
    .sum()
    .reset_index()
)

# 4. Pivot the data so dates are rows and assets are individual columns
pivot_df = daily_energy.pivot(
    index="date", columns="active_assets", values="total_system_kWh"
).fillna(0)

# Include ONLY Grid and Generator 1
hue_order = ["Grid", "Generator 1"]
available_cols = [col for col in hue_order if col in pivot_df.columns]
pivot_df = pivot_df[available_cols]

# Apply your unified corporate color palette
custom_palette = {
    "Grid": "#1f77b4",          # Blue
    "Generator 1": "#ff7f0e",   # Orange
}
colors = [custom_palette[col] for col in pivot_df.columns]

# 5. Generate the Stacked Bar Plot
fig, ax = plt.subplots(figsize=(12, 6))

# Format the date index strings to look clean on the X-axis (e.g., "Mon, Oct 24")
pivot_df.index = pd.to_datetime(pivot_df.index).strftime("%a, %b %d")

pivot_df.plot(
    kind="bar",
    stacked=True,
    color=colors,
    width=0.6,
    edgecolor="black",
    linewidth=0.8,
    alpha=0.85,
    ax=ax,
)

# --- HIDE 0 VALUES COMPLETELY ---
for container in ax.containers:
    # Only format the string if the height is strictly greater than 0
    labels = [f"{v.get_height():,.0f}" if v.get_height() > 0 else "" for v in container]

    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=9,
        fontweight="bold",
        color="white"
    )
# ------------------------------------------

# --- INTEGRATED: ADD TOTAL ENERGY SUM LABELS ON TOP OF BARS ---
# Calculate the combined daily totals and find the maximum peak for scaling the label buffer
daily_totals = pivot_df.sum(axis=1)
max_daily_total = daily_totals.max()

for i, total in enumerate(daily_totals):
    if total > 0:
        ax.text(
            i,
            total + (max_daily_total * 0.015), # 1.5% vertical padding cushion above the bar edge
            f"{total:,.0f}",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
            color="#333333"                     # Premium dark gray color for clear readability
        )
# --------------------------------------------------------------

# Descriptive labels and text formatting
ax.set_title(
    "Daily Energy Consumption Profile by Power Source",
    fontsize=14,
    pad=15,
    weight="bold",
)
ax.set_xlabel("Date", fontsize=11, labelpad=10)
ax.set_ylabel("Total Energy Consumed (kWh)", fontsize=11, labelpad=10)

# Keep X-axis labels completely straight
plt.xticks(rotation=0, ha="center")

# Remove the top and right borders (spines)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Clean up the legend layout
ax.legend(
    title="System State",
    loc="upper right",
    frameon=True,
    facecolor="white",
    edgecolor="none",
)

plt.tight_layout()

# Save the visualization
start_str = pd.to_datetime(daily_energy["date"].min()).strftime("%Y-%m-%d")
end_str = pd.to_datetime(daily_energy["date"].max()).strftime("%Y-%m-%d")
plt.savefig(f"Daily_Energy_Stacked_Source_{start_str}_to_{end_str}.png", dpi=300)

plt.show()

Work on this later

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 1. Clean data and compute the instantaneous power factor
valid_data = abmf_hoffice_24_30_df.copy()

# Remove sensor anomalies and ensure valid power bounds
valid_data = valid_data[
    (valid_data["apparent_power_overall_total"] > 0)
    & (valid_data["load"] <= valid_data["apparent_power_overall_total"])
    & (valid_data["load"] >= 0)
].copy()

valid_data["pf"] = (
    valid_data["load"] / valid_data["apparent_power_overall_total"]
)

# 2. Filter to only look at active operational assets
assets_to_plot = ["Grid", "Generator 1"]
df_plot = valid_data[valid_data["active_assets"].isin(assets_to_plot)].copy()

# 3. Setup the visualization canvas
fig, ax = plt.subplots(figsize=(11, 6))

custom_palette = {
    "Grid": "#1f77b4",        # Blue
    "Generator 1": "#ff7f0e"  # Orange
}

# 4. Generate the scatter plot
# Using small marker size (s) and alpha transparency to handle dense datasets
sns.scatterplot(
    data=df_plot,
    x="load",
    y="pf",
    hue="active_assets",
    palette=custom_palette,
    alpha=0.4,
    s=15,
    edgecolor=None,
    ax=ax
)

# 5. Styling and annotations
ax.set_title(
    "System Power Factor vs. Active Load Profile",
    fontsize=14,
    pad=20,
    weight="bold",
)
ax.set_xlabel("Active Load (kW)", fontsize=11, labelpad=10)
ax.set_ylabel("Power Factor (kW / kVA)", fontsize=11, labelpad=10)

# Clamp the Y-axis strictly between 0 and a bit over 1.0 for PF
ax.set_ylim(0, 1.02)

# Subtle grid lines are crucial for scatter plots to map coordinate intersections
# ax.grid(True, linestyle=":", alpha=0.5)

# Clean up borders (spines)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Clean up legend container padding and background
ax.legend(
    title="Power Source",
    loc="lower right",
    frameon=True,
    facecolor="white",
    edgecolor="none"
)

plt.tight_layout()

# Save the visualization
plt.savefig("power_factor_vs_load_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Clean data and filter anomalies
valid_data = abmf_hoffice_24_30_df.copy()
valid_data = valid_data[
    (valid_data["apparent_power_overall_total"] > 0)
    & (valid_data["load"] <= valid_data["apparent_power_overall_total"])
    & (valid_data["load"] >= 0)
].copy()

# 2. Calculate Reactive Power (kVAR) using Q = sqrt(S^2 - P^2)
valid_data["reactive_power"] = np.sqrt(
    np.maximum(0, valid_data["apparent_power_overall_total"]**2 - valid_data["load"]**2)
)

# 3. Split data by asset for side-by-side tracking
grid_data = valid_data[valid_data["active_assets"] == "Grid"]
gen_data = valid_data[valid_data["active_assets"] == "Generator 1"]

# 4. Create side-by-side subplots sharing axes for perfect scale comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

# Define clean alpha transparency and size to handle the heavy minute-log density
alpha_val = 0.35
marker_size = 12

# Left Panel: Grid capacity breakdown
ax1.scatter(grid_data["load"], grid_data["apparent_power_overall_total"],
            color="#8e44ad", alpha=alpha_val, s=marker_size, label="Apparent Power (kVA)", edgecolor='none')
ax1.scatter(grid_data["load"], grid_data["reactive_power"],
            color="#e74c3c", alpha=alpha_val, s=marker_size, label="Reactive Power (kVAR)", edgecolor='none')
ax1.set_title("Grid Capacity Utilization", fontsize=12, pad=10)
ax1.grid(True, linestyle=":", alpha=0.5)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# Right Panel: Generator capacity breakdown
ax2.scatter(gen_data["load"], gen_data["apparent_power_overall_total"],
            color="#8e44ad", alpha=alpha_val, s=marker_size, label="Apparent Power (kVA)", edgecolor='none')
ax2.scatter(gen_data["load"], gen_data["reactive_power"],
            color="#e74c3c", alpha=alpha_val, s=marker_size, label="Reactive Power (kVAR)", edgecolor='none')
ax2.set_title("Generator 1 Capacity Utilization", fontsize=12, pad=10)
ax2.grid(True, linestyle=":", alpha=0.5)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# 5. Add the Ideal PF = 1.0 Reference Line (Y = X)
# On a perfect system, kVA completely matches kW, and kVAR is exactly zero.
max_kw = valid_data["load"].max()
ref_x = np.linspace(0, max_kw * 1.05, 100)

for ax in [ax1, ax2]:
    ax.plot(ref_x, ref_x, color="black", linestyle="--", linewidth=1.5,
            label="Ideal Border (PF = 1.0: kVA = kW)")
    ax.set_xlim(-0.5, max_kw * 1.05)
    ax.set_ylim(-0.5, valid_data["apparent_power_overall_total"].max() * 1.05)

# Deduplicate and attach the clean legends
handles, labels = ax1.get_legend_handles_labels()
ax1.legend(handles, labels, loc="upper left", frameon=True, facecolor="white", edgecolor="none")
ax2.legend(handles, labels, loc="upper left", frameon=True, facecolor="white", edgecolor="none")

# 6. Apply global layout parameters and padded text labels
fig.suptitle("Delivered Capacity (kVA & kVAR) vs. Useful Active Load (kW)",
             fontsize=14, fontweight="bold", y=0.98)
fig.supxlabel("Active Load (kW)", fontsize=11, y=0.02)
fig.supylabel("Total / Reactive Demand (kVA / kVAR)", fontsize=11, x=0.01)

# Ensure no labels are clipped during export
plt.tight_layout(pad=3.0)

# Save the visualization directly
plt.savefig("capacity_vs_load_power_triangle.png", dpi=300, bbox_inches="tight")

Update from Elie

In [ ]:
abmf_hoffice_24_30_df.columns

In [ ]:
abmf_hoffice_24_30_df[abmf_hoffice_24_30_df['current_unbalance_factor'] > 190]

In [ ]:
# 1. Filter the dataframe based on your criteria
# (Make sure your time/date column is converted to datetime first)
high_unbalance_df = abmf_hoffice_24_30_df[abmf_hoffice_24_30_df['current_unbalance_factor'] > 190].copy()

# 2. Convert your timestamp column to datetime if you haven't already
# Replace 'timestamp_column' with the actual name of your time column (e.g., 'DateTime', 'time', 'Timestamp')
high_unbalance_df['timestamp'] = pd.to_datetime(high_unbalance_df['timestamp'])

# 3. Extract the hour of the day to see a daily distribution pattern
high_unbalance_df['hour'] = high_unbalance_df['timestamp'].dt.hour

# 4. Create the visual
plt.figure(figsize=(10, 6))
sns.histplot(data=high_unbalance_df, x='hour', bins=24, kde=True, color='crimson')

# Formatting the plot
plt.title('Distribution of High Current Unbalance Factor (> 190) by Hour of Day', fontsize=14, pad=15)
plt.xlabel('Hour of Day (0-23)', fontsize=12)
plt.ylabel('Frequency of Occurrence', fontsize=12)
plt.xticks(range(0, 24))
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
abmf_hoffice_24_30_df.columns

In [ ]:
abmf_hoffice_24_30_df[(abmf_hoffice_24_30_df['current_unbalance_factor'] > 190) & (abmf_hoffice_24_30_df['active_assets'] == 'Grid')].copy()

In [ ]:
abmf_hoffice_24_30_df[(abmf_hoffice_24_30_df['current_unbalance_factor'] > 190) & (abmf_hoffice_24_30_df['active_assets'] == 'Generator 1')].copy()

In [ ]:
abmf_hoffice_24_30_df[(abmf_hoffice_24_30_df['current_unbalance_factor'] > 190)].groupby('active_assets').size()

In [ ]:
abmf_hoffice_24_30_df[(abmf_hoffice_24_30_df['current_unbalance_factor'] > 190) & (abmf_hoffice_24_30_df['active_assets'] == '')].copy()

Note the above

In [ ]:
abmf_hoffice_24_30_df[(abmf_hoffice_24_30_df['current_unbalance_factor'] > 190) & (abmf_hoffice_24_30_df['active_assets'].isin(['Generator 1', 'Grid']))]['load'].agg(['mean', 'min', 'max'])

In [ ]:
abmf_hoffice_24_30_df[(abmf_hoffice_24_30_df['current_unbalance_factor'] > 100) & (abmf_hoffice_24_30_df['current_unbalance_factor'] < 125)]

In [ ]:
# 1. Filter the data based on your threshold criteria
high_unbalance_df = abmf_hoffice_24_30_df[(abmf_hoffice_24_30_df['current_unbalance_factor'] > 190) & (abmf_hoffice_24_30_df['active_assets'] == 'Grid')].copy()

# 2. Ensure your time column is parsed correctly as a datetime
# !! REPLACE 'timestamp_column' with the actual name of your time/date column !!
high_unbalance_df['timestamp'] = pd.to_datetime(high_unbalance_df['timestamp'])

# 3. Extract Day of Week and Hour of Day
high_unbalance_df['day_of_week'] = high_unbalance_df['timestamp'].dt.day_name()
high_unbalance_df['hour'] = high_unbalance_df['timestamp'].dt.hour

# 4. Create a pivot table counting occurrences for every Day/Hour combination
# This forms the underlying matrix for the heatmap
heatmap_data = high_unbalance_df.groupby(['day_of_week', 'hour']).size().unstack(fill_value=0)

# 5. Order the days properly so they don't sort alphabetically
days_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
heatmap_data = heatmap_data.reindex(days_order)

# 6. Ensure all 24 hours (0-23) are represented in the columns, even if some have 0 counts
heatmap_data = heatmap_data.reindex(columns=range(0, 24), fill_value=0)

# 7. Plot the Heatmap
plt.figure(figsize=(14, 7))
sns.heatmap(
    heatmap_data,
    cmap='YlOrRd',    # Yellow to Orange to Red color gradient for frequency
    annot=True,       # Displays the actual count inside each block
    fmt='d',          # Format numbers as integers
    linewidths=0.5,   # Subtle lines separating the blocks
    cbar_kws={'label': 'Count of Occurrences (> 190)'}
)

# Formatting the visual
plt.title('Weekly Routine of High Current Unbalance (> 190)\nDistribution by Day of Week and Hour', fontsize=14, pad=15)
plt.xlabel('Hour of Day (0-23)', fontsize=12)
plt.ylabel('Day of Week', fontsize=12)
plt.xticks(rotation=0) # Keep hour labels horizontal for readability

plt.tight_layout()
plt.show()

In [ ]:
# 1. Create the plot layout
plt.figure(figsize=(10, 6))

# 2. Generate a scatter plot
# (Replace 'load_column' with the exact name of your load column)
sns.scatterplot(
    data=abmf_hoffice_24_30_df,
    x='load',
    y='current_unbalance_factor',
    alpha=0.5,          # Makes points semi-transparent to see density where data overlaps
    color='teal'
)

# 3. Add a horizontal reference line at your threshold (190) to see where the problem zone lives
plt.axhline(y=190, color='crimson', linestyle='--', linewidth=1.5, label='Threshold (190)')

# Formatting the visual
plt.title('System Load vs. Current Unbalance Factor', fontsize=14, pad=15)
plt.xlabel('Load (Independent Driver)', fontsize=12)
plt.ylabel('Current Unbalance Factor (Dependent Outcome)', fontsize=12)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Filter data to include ONLY Grid and Generator 1
filtered_df = abmf_hoffice_24_30_df[abmf_hoffice_24_30_df['active_assets'].isin(['Grid', 'Generator 1'])].copy()

plt.figure(figsize=(11, 7))

# 2. Plot using 'hue' to automatically color-code and 'style' to change markers
sns.scatterplot(
    data=filtered_df,
    x='load',  # Replace with your actual load column name
    y='current_unbalance_factor',
    hue='active_assets',
    style='active_assets',
    palette={'Grid': '#1f77b4', 'Generator 1': '#ff7f0e'}, # Crisp blue and orange
    alpha=0.6,
    s=60 # Point size
)

# 3. Add the threshold reference line
plt.axhline(y=190, color='crimson', linestyle='--', linewidth=1.5, label='Critical Threshold (190)')

# Formatting
plt.title('Load vs. Current Unbalance: Grid vs. Generator 1 Performance', fontsize=14, pad=15)
plt.xlabel('System Load', fontsize=12)
plt.ylabel('Current Unbalance Factor', fontsize=12)
plt.legend(title='Active Asset / Reference')
plt.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Filter data to include ONLY Grid and Generator 1
filtered_df = abmf_hoffice_24_30_df[abmf_hoffice_24_30_df['active_assets'].isin(['Grid', 'Generator 1'])].copy()

# 2. Convert to datetime and extract the hour (Replace 'timestamp' with your actual column name)
filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
filtered_df['hour'] = filtered_df['timestamp'].dt.hour

# 3. Filter for working hours: 8 AM (hour 8) to 5 PM (hour 17) inclusive
working_hours_df = filtered_df[(filtered_df['hour'] >= 8) & (filtered_df['hour'] <= 17)]

# 4. Initialize the figure
plt.figure(figsize=(11, 7))

# 5. Plot using the working hours dataframe
sns.scatterplot(
    data=working_hours_df,
    x='load',
    y='current_unbalance_factor',
    hue='active_assets',
    style='active_assets',
    palette={'Grid': '#1f77b4', 'Generator 1': '#ff7f0e'},
    alpha=0.6,
    s=60
)

# 6. Add the threshold reference line
plt.axhline(y=190, color='crimson', linestyle='--', linewidth=1.5, label='Critical Threshold (190)')

# Formatting
plt.title('Load vs. Current Unbalance: Grid vs. Generator 1\n(Working Hours: 8:00 AM - 5:00 PM)', fontsize=14, pad=15)
plt.xlabel('System Load', fontsize=12)
plt.ylabel('Current Unbalance Factor', fontsize=12)
plt.legend(title='Active Asset / Reference')
plt.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
plt.style.use('default')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Reset to clean white default theme to prevent black backgrounds
plt.style.use('default')

# 1. Filter data to include ONLY Grid and Generator 1
filtered_df = abmf_hoffice_24_30_df[abmf_hoffice_24_30_df['active_assets'].isin(['Grid', 'Generator 1'])].copy()

# 2. Convert to datetime and extract Hour and Day Name
filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
filtered_df['hour'] = filtered_df['timestamp'].dt.hour
filtered_df['day_name'] = filtered_df['timestamp'].dt.day_name()

# 3. Filter for working hours: 8 AM (hour 8) to 5 PM (hour 17) inclusive
time_filtered_df = filtered_df[(filtered_df['hour'] >= 8) & (filtered_df['hour'] <= 17)]

# 4. Keep ONLY Monday, Tuesday, and Friday
target_days = ['Monday', 'Tuesday', 'Friday']
final_filtered_df = time_filtered_df[time_filtered_df['day_name'].isin(target_days)]

# 5. Initialize the figure
plt.figure(figsize=(11, 7))

# 6. Plot using explicit markers, custom colors, and fixed mapping
sns.scatterplot(
    data=final_filtered_df,
    x='load',
    y='current_unbalance_factor',
    hue='active_assets',
    style='active_assets',
    hue_order=['Grid', 'Generator 1'],                  # Enforces Grid layout order
    style_order=['Grid', 'Generator 1'],                # Enforces Grid marker order
    markers={'Grid': 'o', 'Generator 1': 'X'},          # Circle for Grid, X for Gen 1
    palette={'Grid': '#1f77b4', 'Generator 1': '#ff7f0e'},
    alpha=0.6,
    s=60
)

# 7. Add the threshold reference line
plt.axhline(y=190, color='crimson', linestyle='--', linewidth=1.5, label='Critical Threshold (190)')

# Formatting
plt.title('Load vs. Current Unbalance: Grid vs. Generator 1\n(Mon, Tue, Fri Only | 8:00 AM - 5:00 PM)', fontsize=14, pad=15)
plt.xlabel('System Load', fontsize=12)
plt.ylabel('Current Unbalance Factor', fontsize=12)

# Legend configurations matching 'image_fb8f1c.png' exactly
plt.legend(
    title='Active Asset / Reference',
    loc='center right',       # Moves it to the middle right position
    frameon=True,             # Turns on the box frame
    facecolor='white',        # Matches white background
    edgecolor='#cccccc'       # Subtle grey outline for the box
)

# Subtle grid layout matching image style
plt.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Set up a dark background style matching the reference image
plt.style.use('dark_background')

# 2. Filter data to include ONLY Grid and Generator 1
filtered_df = abmf_hoffice_24_30_df[abmf_hoffice_24_30_df['active_assets'].isin(['Grid', 'Generator 1'])].copy()

# 3. Initialize the figure size
plt.figure(figsize=(11, 7))

# 4. Plot using explicit markers and the color palette from the reference image
sns.scatterplot(
    data=filtered_df,
    x='load',
    y='current_unbalance_factor',
    hue='active_assets',
    style='active_assets',
    markers={'Grid': 'o', 'Generator 1': 'x'},          # Circles and X's
    palette={'Grid': '#4682B4', 'Generator 1': '#D57A25'}, # Steel blue and rust orange
    alpha=0.7,
    s=100                                                # Slightly larger point size for high visibility
)

# 5. Add the red dashed threshold reference line at 190
plt.axhline(y=190, color='#C8374D', linestyle='--', linewidth=2, label='Critical Threshold (190)')

# 6. Formatting exactly matching the reference chart
plt.title('Load vs. Current Unbalance: Grid vs. Generator 1 Performance', fontsize=16, pad=15)
plt.xlabel('load (fontsize=12)', fontsize=12)
plt.ylabel('current_unbalance_factor (fontsize=12)', fontsize=12)

# Legend adjustments to match the box profile
plt.legend(title='', facecolor='#1F242D', edgecolor='white', loc='upper right', fontsize=11)

# Subtle dotted grid lines matching the image style
plt.grid(True, linestyle='--', color='#555555', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
abmf_hoffice_24_30_df.sample(5)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

# 1. Plot the raw data points with high transparency (alpha)
# This makes dense clusters darker and single spikes lighter
sns.scatterplot(
    data=abmf_hoffice_24_30_df,
    x='load',
    y='current_unbalance_factor',
    alpha=0.3,
    color='slategray',
    label='Data Points'
)

# 2. Add a trend line to show the overall behavior
# order=2 creates a curved polynomial line to capture non-linear drops
sns.regplot(
    data=abmf_hoffice_24_30_df,
    x='load',
    y='current_unbalance_factor',
    scatter=False,
    order=2,
    color='crimson',
    label='Operational Trend'
)

# 3. Highlight the "Low Load / High Unbalance" risk zone using a shaded box
# Adjust the numbers (e.g., xmax=50, ymin=190) based on your actual data ranges
plt.axvspan(xmin=0, xmax=30, color='red', alpha=0.1, label='High Risk Low-Load Zone')
plt.axhline(y=190, color='black', linestyle=':', alpha=0.7)

# Formatting
plt.title('Evidence of Inverse Relationship: Highest Current Unbalance Occurs at Low Loads', fontsize=13, pad=15)
plt.xlabel('System Load', fontsize=11)
plt.ylabel('Current Unbalance Factor', fontsize=11)
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Creates a joint scatter + histogram plot
g = sns.jointplot(
    data=abmf_hoffice_24_30_df,
    x='load',
    y='current_unbalance_factor',
    kind='reg', # Automatically adds a scatter plot and a regression trend line
    color='teal',
    height=8,
    scatter_kws={'alpha': 0.3},
    line_kws={'color': 'crimson', 'linewidth': 2}
)

# Super title adjustment for joint plots
g.fig.suptitle('Distribution Analysis: Current Unbalance Peaks under Low Load Conditions', y=1.02, fontsize=14)
g.set_axis_labels('System Load', 'Current Unbalance Factor', fontsize=12)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Create a copy of the dataframe to work with
heatmap_avg_df = abmf_hoffice_24_30_df.copy()

# 2. Ensure your time column is parsed correctly as a datetime
# !! REPLACE 'timestamp_column' with the actual name of your time/date column !!
heatmap_avg_df['timestamp'] = pd.to_datetime(heatmap_avg_df['timestamp'])

# 3. Extract Day of Week and Hour of Day
heatmap_avg_df['day_of_week'] = heatmap_avg_df['timestamp'].dt.day_name()
heatmap_avg_df['hour'] = heatmap_avg_df['timestamp'].dt.hour

# 4. Pivot the data using the MEAN (average) instead of size/count
# .pivot_table handles the aggregation automatically via aggfunc='mean'
heatmap_data_avg = heatmap_avg_df.pivot_table(
    index='day_of_week',
    columns='hour',
    values='current_unbalance_factor',
    aggfunc='mean'
)

# 5. Order the days properly so they read chronologically Monday -> Sunday
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data_avg = heatmap_data_avg.reindex(days_order)

# 6. Ensure all 24 hours (0-23) are explicitly represented
heatmap_data_avg = heatmap_data_avg.reindex(columns=range(0, 24))

# 7. Plot the Average Heatmap
plt.figure(figsize=(15, 7))
sns.heatmap(
    heatmap_data_avg,
    cmap='YlGnBu',     # A clean Yellow -> Green -> Blue palette to differentiate from the count chart
    annot=True,        # Displays the actual average value inside each block
    fmt='.1f',         # Format numbers to 1 decimal place (e.g., 45.3)
    linewidths=0.5,    # Subtle separation lines
    cbar_kws={'label': 'Average Current Unbalance Factor'}
)

# Formatting the visual
plt.title('Weekly Baseline Profile: Mean Current Unbalance Factor by Time', fontsize=14, pad=15)
plt.xlabel('Hour of Day (0-23)', fontsize=12)
plt.ylabel('Day of Week', fontsize=12)
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Enforce clean default layout theme
plt.style.use('default')

# 1. Clean data: Drop missing values and filter out negative anomalies
valid_data = abmf_hoffice_24_30_df.dropna(subset=["current_unbalance_factor"])
valid_data = valid_data[valid_data["current_unbalance_factor"] >= 0].copy()

# --- TIME & DAY FILTER IMPLEMENTATION ---
# Convert to datetime and extract Hour and Day Name
valid_data['timestamp'] = pd.to_datetime(valid_data['timestamp'])
valid_data['hour'] = valid_data['timestamp'].dt.hour
valid_data['day_name'] = valid_data['timestamp'].dt.day_name()

# Filter for working hours: 8 AM (hour 8) to 5 PM (hour 17) inclusive
time_filtered_data = valid_data[(valid_data['hour'] >= 8) & (valid_data['hour'] <= 17)]

# Keep ONLY Monday, Tuesday, and Friday
target_days = ['Monday', 'Tuesday', 'Friday']
final_filtered_data = time_filtered_data[time_filtered_data['day_name'].isin(target_days)]
# ----------------------------------------

# 2. Auto-detect scale (Fraction vs. Percentage) & setup appropriate bins
max_val = final_filtered_data["current_unbalance_factor"].max()

bin_step = 5  # 5% granularity
x_label = "Current Unbalance Factor (%)"
x_max = max(10.0, max_val * 1.1)  # Focus window, default to at least 10% unbalance

bins = np.arange(0, x_max + (bin_step * 2), bin_step)
centers = (bins[:-1] + bins[1:]) / 2
bar_width = bin_step * 0.8

# 3. Extract data splits per asset from the filtered timeframe
grid_cuf = final_filtered_data[final_filtered_data["active_assets"] == "Grid"]["current_unbalance_factor"]
gen_cuf = final_filtered_data[final_filtered_data["active_assets"] == "Generator 1"]["current_unbalance_factor"]

# Compute distributions
grid_counts, _ = np.histogram(grid_cuf, bins=bins)
gen_counts, _ = np.histogram(gen_cuf, bins=bins)

# Convert minute-log counts to hours
grid_hours = grid_counts / 60.0
gen_hours = gen_counts / 60.0

# 4. Create side-by-side plots sharing the Y-axis (Hours)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

# Left Panel: Grid Current Unbalance Distribution
ax1.bar(
    centers,
    grid_hours,
    width=bar_width,
    color="#1f77b4",
    edgecolor="black",
    alpha=0.85,
)
ax1.set_title("Grid Current Unbalance Profile", fontsize=12)
ax1.set_ylabel("Operation Time (Hours)", fontsize=11)
ax1.set_xlim(0, x_max)
ax1.grid(False)

# Remove the top and right borders (spines)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# Right Panel: Generator Current Unbalance Distribution
ax2.bar(
    centers,
    gen_hours,
    width=bar_width,
    color="#ff7f0e",
    edgecolor="black",
    alpha=0.85,
)
ax2.set_title("Generator 1 Current Unbalance Profile", fontsize=12)
ax2.set_xlim(0, x_max)
ax2.grid(False)

# Remove the top and right borders (spines)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# --- GLOBAL LABELS ---
# Shared Figure Main Title (Updated to reflect specific timeline)
fig.suptitle(
    "Current Unbalance Factor Distribution\n(Mon, Tue, Fri Only | 8:00 AM - 5:00 PM)",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

# Shared Figure X-Axis Label (Centered across both plots)
fig.supxlabel(x_label, fontsize=11, y=0.02)
# ----------------------------

plt.tight_layout()
# plt.savefig("current_unbalance_histograms_filtered_time.png", bbox_inches="tight")
plt.show()

I need to fill the empty spaces here first

In [ ]:
df_filled

In [ ]:
# turn 'timestamp' into a regular column
df = df_filled.reset_index()

# Rename index into timestamp if not currently nameed timestamp
if "timestamp" not in df.columns and "index" in df.columns:
    df = df.rename(columns={"index": "timestamp"})

# Clean, sort, and parse data
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Replace empty strings and NaNs with clear tracking labels
df["active_assets"] = df["active_assets"].replace(
    {"": "None / Standby", np.nan: "No Power"}
)

# Fill NaN load values with 0 to show a visible line during outages
# df["load"] = df["load"].fillna(0)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Enforce clean default layout theme
plt.style.use('default')

# Your exact original color palette mapping
custom_palette = {
    "Grid": "#1f77b4",
    "Generator 1": "#ff7f0e",
    "None / Standby": "#7f7f7f",
    "No Power": "#000000",
}
expected_categories = list(custom_palette.keys())

# =====================================================================
# 1. CALCULATE PERCENTAGES FOR BOTH CATEGORIES WITH ZERO-FILLS
# =====================================================================

# --- A. Calculate Total Percentages (All days, all hours) ---
total_counts = df['active_assets'].value_counts()
total_percentages = (total_counts / total_counts.sum()) * 100
total_percentages = total_percentages.reindex(expected_categories, fill_value=0)

# --- B. Calculate Working Hours Percentages (Mon, Tue, Fri | 8 AM - 5 PM) ---
working_df = df.reset_index().copy()
working_df['timestamp'] = pd.to_datetime(working_df['timestamp'])
working_df['hour'] = working_df['timestamp'].dt.hour
working_df['day_name'] = working_df['timestamp'].dt.day_name()

# Apply the strict time and day filters
time_mask = (working_df['hour'] >= 8) & (working_df['hour'] <= 17)
day_mask = working_df['day_name'].isin(['Monday', 'Tuesday', 'Friday'])
working_filtered_df = working_df[time_mask & day_mask]

working_counts = working_filtered_df['active_assets'].value_counts()
working_hours_percentages = (working_counts / working_counts.sum()) * 100
working_hours_percentages = working_hours_percentages.reindex(expected_categories, fill_value=0)


# =====================================================================
# 2. STRUCTURE DATA FOR THE COMBO BAR CHART
# =====================================================================

df_total = pd.DataFrame({
    'Source Type': total_percentages.index,
    'Percentage (%)': total_percentages.values,
    'Timeframe': 'Total'
})

df_working = pd.DataFrame({
    'Source Type': working_hours_percentages.index,
    'Percentage (%)': working_hours_percentages.values,
    'Timeframe': 'Working Hours'
})

combo_df = pd.concat([df_total, df_working], ignore_index=True)


# =====================================================================
# 3. BUILD THE VISUAL WITH EXPLICIT SIDE-BY-SIDE GROUPING
# =====================================================================

plt.figure(figsize=(13, 6))

# Group by Source Type along X, and split side-by-side using Timeframe as Hue
ax = sns.barplot(
    data=combo_df,
    x='Source Type',
    y='Percentage (%)',
    hue='Timeframe',
    edgecolor="black",
    linewidth=0.7
)

ax.grid(False)

# --- APPLY THE UPDATED SHADING HIERARCHY ---
# First 4 patches are 'Total' bars, next 4 patches are 'Working Hours' bars.
for i, asset in enumerate(expected_categories):
    base_color = custom_palette[asset]

    # 'Total' baseline bars get the LIGHTER shade (alpha = 0.45)
    ax.patches[i].set_facecolor(base_color)
    ax.patches[i].set_alpha(0.45)

    # 'Working Hours' focus bars get the SOLID, DARKER shade (alpha = 1.0)
    ax.patches[i + len(expected_categories)].set_facecolor(base_color)
    ax.patches[i + len(expected_categories)].set_alpha(1.0)

# --- CLEAN UP BORDERS (SPINES) ---
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Add the integer percentage labels cleanly above each bar
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=10)

# Labels and Formatting
plt.title("Power Source Distribution: Total vs. Working Hours Comparison", fontsize=14, pad=15)
plt.xlabel("Source Type", fontsize=12)
plt.ylabel("Percentage (%)", fontsize=12)
plt.ylim(0, 110)

# Updated manual legend to reflect the light-to-dark shift
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#555555', edgecolor='black', alpha=0.45, label='Total Hours'),
    Patch(facecolor='#555555', edgecolor='black', alpha=1.0, label='Working Hours')
]
plt.legend(handles=legend_elements, title="Analysis Framework", loc="upper right", frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
df_total

In [ ]:
df_working

In [ ]:
abmf_hoffice_24_30_df_copy = abmf_hoffice_24_30_df.copy()

In [ ]:
# 1. Convert the column to datetime (just in case it loaded as a string/object)
abmf_hoffice_24_30_df_copy["timestamp"] = pd.to_datetime(abmf_hoffice_24_30_df_copy["timestamp"])

# 2. Set the column as the index
abmf_hoffice_24_30_df_copy = abmf_hoffice_24_30_df_copy.set_index("timestamp")

# 3. Crucial: Sort the index chronologically
abmf_hoffice_24_30_df_copy = abmf_hoffice_24_30_df_copy.sort_index()

In [ ]:
df_filled = abmf_hoffice_24_30_df_copy.resample("1min").first()

18/06

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Enforce clean default layout theme
plt.style.use('default')

# =====================================================================
# 1. APPLY WORKING HOURS & DAY FILTERS FIRST
# =====================================================================
working_df = df.reset_index().copy()
working_df['timestamp'] = pd.to_datetime(working_df['timestamp'])
working_df['hour'] = working_df['timestamp'].dt.hour
working_df['day_name'] = working_df['timestamp'].dt.day_name()

# Apply strict filters: 8 AM to 5 PM inclusive, on Mon, Tue, Fri
time_mask = (working_df['hour'] >= 8) & (working_df['hour'] <= 17)
day_mask = working_df['day_name'].isin(['Monday', 'Tuesday', 'Friday'])
time_filtered_df = working_df[time_mask & day_mask]


# =====================================================================
# 2. CLEAN, GROUP, AND PROCESS DATA
# =====================================================================
# Filter out 'No Power' and 'None / Standby' before grouping
df_filtered = time_filtered_df[~time_filtered_df["active_assets"].isin(["No Power", "None / Standby"])]

# Group by active_assets and sum total_system_kWh
grouped_df = (
    df_filtered.groupby("active_assets")["total_system_kWh"].sum().reset_index()
)

# Convert to percentages based on the remaining active assets
total_kWh = grouped_df["total_system_kWh"].sum()
grouped_df["percentage"] = (grouped_df["total_system_kWh"] / total_kWh) * 100

# Sort the values in descending order
grouped_df = grouped_df.sort_values(by="percentage", ascending=False)


# =====================================================================
# 3. DEFINE PALETTE & GENERATE DOUGHNUT CHART
# =====================================================================
# Define the custom color scheme
custom_palette = {
    "Grid": "#1f77b4",  # Blue
    "Generator 1": "#ff7f0e",  # Orange
}

colors = [
    custom_palette.get(asset, "#7f7f7f") for asset in grouped_df["active_assets"]
]

# Generate the Doughnut Chart
fig, ax = plt.subplots(figsize=(6, 5))

wedges, texts, autotexts = ax.pie(
    grouped_df["percentage"],
    labels=grouped_df["active_assets"],
    autopct="%.1f%%",
    startangle=90,
    colors=colors,
    # width=0.4 punches out the center hole
    wedgeprops=dict(edgecolor="black", linewidth=0.8, width=0.4),
    textprops=dict(fontsize=11),
    pctdistance=0.8  # Pushes the text outward to center it in the ring
)

# Style the percentage text inside the ring for high contrast
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_weight('bold')

# Styling and formatting adjustments reflecting the specific timeline
ax.set_title(
    "Working Hours System Consumption by Active Assets (%)\n(Mon, Tue, Fri Only | 8:00 AM - 5:00 PM)",
    fontsize=12,
    fontweight="bold",
    pad=15,
)

plt.tight_layout()
plt.savefig("working_hours_system_doughnut_chart.png", bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Enforce clean default layout theme
plt.style.use('default')

# =====================================================================
# 1. APPLY WORKING HOURS & DAY FILTERS FIRST
# =====================================================================
working_df = abmf_hoffice_24_30_df.reset_index().copy()
working_df['timestamp'] = pd.to_datetime(working_df['timestamp'])
working_df['hour'] = working_df['timestamp'].dt.hour
working_df['day_name'] = working_df['timestamp'].dt.day_name()

# Apply strict filters: 8 AM to 5 PM inclusive, on Mon, Tue, Fri
time_mask = (working_df['hour'] >= 8) & (working_df['hour'] <= 17)
day_mask = working_df['day_name'].isin(['Monday', 'Tuesday', 'Friday'])
time_filtered_df = working_df[time_mask & day_mask]


# =====================================================================
# 2. TARGET ASSETS AND CALCULATE AGGREGATED METRICS
# =====================================================================
# Target your specific generator and grid assets
assets_to_plot = ["Grid", "Generator 1"]
filtered_df = time_filtered_df[time_filtered_df["active_assets"].isin(assets_to_plot)]

# Group by active_assets and calculate min, mean (average), and max simultaneously
summary_df = (
    filtered_df.groupby("active_assets")["load"]
    .agg(["min", "mean", "max"])
    .reset_index()
)

# Sort your categories by their mean load descending
summary_df = summary_df.sort_values(by="mean", ascending=False)


# =====================================================================
# 3. GENERATE THE GROUPED BAR CHART
# =====================================================================
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(summary_df["active_assets"]))
width = 0.22  # Optimal thickness for side-by-side bars

# Shifting from "status/risk" to "magnitude" with your original blues layout
rects_min = ax.bar(x - width, summary_df['min'], width, label='Minimum', color='#a1c4fd')  # Light Blue
rects_avg = ax.bar(x, summary_df['mean'], width, label='Average', color='#4a90e2')       # Medium Blue
rects_max = ax.bar(x + width, summary_df['max'], width, label='Maximum', color='#003366')  # Deep Navy Blue

# Styling, titles, and text label formatting reflecting the filtered timeline
ax.set_title(
    "Working Hours Load Analysis: Min, Avg, and Max Load by Asset\n(Mon, Tue, Fri Only | 8:00 AM - 5:00 PM)",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
ax.set_xlabel("Active Assets", fontsize=12, labelpad=10)
ax.set_ylabel("Load (kW)", fontsize=12, labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels(summary_df["active_assets"], fontsize=11)
ax.legend(fontsize=11)

# Keep the background clean with no grid lines
ax.grid(False)

# Remove the top and right borders (spines)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Add precise value labels on top of every single bar with proper mathematical rounding
ax.bar_label(rects_min, padding=3, fmt="%.0f kW", fontsize=9)
ax.bar_label(rects_avg, padding=3, fmt="%.0f kW", fontsize=9, fontweight="bold")
ax.bar_label(rects_max, padding=3, fmt="%.0f kW", fontsize=9)

# Expand the y-axis ceiling slightly so the text labels don't get squished
if not summary_df.empty:
    ax.set_ylim(0, max(summary_df["max"]) * 1.15)

# Save visualization layout cleanly
plt.tight_layout()
plt.savefig("working_hours_generator_grid_load_summary.png", bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Enforce clean default layout theme
plt.style.use('default')

# =====================================================================
# 1. APPLY WORKING HOURS & DAY FILTERS FIRST
# =====================================================================
working_df = abmf_hoffice_24_30_df.reset_index().copy()
working_df['timestamp'] = pd.to_datetime(working_df['timestamp'])
working_df['hour'] = working_df['timestamp'].dt.hour
working_df['day_name'] = working_df['timestamp'].dt.day_name()

# Apply strict filters: 8 AM to 5 PM inclusive, on Mon, Tue, Fri
time_mask = (working_df['hour'] >= 8) & (working_df['hour'] <= 17)
day_mask = working_df['day_name'].isin(['Monday', 'Tuesday', 'Friday'])
time_filtered_df = working_df[time_mask & day_mask]


# =====================================================================
# 2. CLEAN DATA & EXTRACT METRICS (FROM FILTERED WINDOW)
# =====================================================================
# Clean data: Filter out impossible sensor anomalies
valid_data = time_filtered_df[
    (time_filtered_df["apparent_power_overall_total"] > 0)
    & (time_filtered_df["load"] <= time_filtered_df["apparent_power_overall_total"])
].copy()

# Filter exclusively for Grid and Generator assets
target_assets = ["Grid", "Generator 1"]
valid_data = valid_data[valid_data["active_assets"].isin(target_assets)]

# Calculate instantaneous power factor per minute for MIN and MAX extraction
valid_data["pf_instantaneous"] = (
    valid_data["load"] / valid_data["apparent_power_overall_total"]
)

# Extract Min and Max boundaries
min_max_df = (
    valid_data.groupby("active_assets")["pf_instantaneous"]
    .agg(["min", "max"])
    .reset_index()
)

# Extract sums for the true Energy-Weighted Average
sums_df = (
    valid_data.groupby("active_assets")[
        ["load", "apparent_power_overall_total"]
    ]
    .sum()
    .reset_index()
)

# Merge metrics and calculate the final weighted mean
summary_df = pd.merge(min_max_df, sums_df, on="active_assets")
summary_df["mean"] = (
    summary_df["load"] / summary_df["apparent_power_overall_total"]
)

# Sort by average performance for visualization layout
summary_df = summary_df.sort_values(by="mean", ascending=False)


# =====================================================================
# 3. CONSTRUCT GROUPED BAR CHART
# =====================================================================
fig, ax = plt.subplots(figsize=(10, 5.5))

x = np.arange(len(summary_df["active_assets"]))
width = 0.22  # Balanced spacing for 3 metrics side-by-side

# Neutral, modern color palette
c_min = "#d32f2f"
c_avg = "#4a90e2"
c_max = "#1c3d5a"

# Plot Min, Avg, Max bars
rects_min = ax.bar(
    x - width,
    summary_df["min"],
    width,
    label="Minimum",
    color=c_min,
    edgecolor="black",
)
rects_avg = ax.bar(
    x,
    summary_df["mean"],
    width,
    label="Weighted Average",
    color=c_avg,
    edgecolor="black",
)
rects_max = ax.bar(
    x + width,
    summary_df["max"],
    width,
    label="Maximum",
    color=c_max,
    edgecolor="black",
)

# Styling and Labels updated for the targeted time window
ax.set_title(
    "Working Hours Power Factor Analysis: Min, Weighted Avg, and Max by Asset\n(Mon, Tue, Fri Only | 8:00 AM - 5:00 PM)",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
ax.set_xlabel("Active Assets", fontsize=12, labelpad=10)
ax.set_ylabel("Power Factor", fontsize=12, labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels(summary_df["active_assets"], fontsize=11)
ax.set_ylim(0, 1.18)  # Room for labels without ceiling truncation
ax.legend(fontsize=11, loc="upper left")

# Remove the top and right borders (spines)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Remove background gridlines entirely
ax.grid(False)

# Add clear 2-decimal values on top of every bar
ax.bar_label(rects_min, padding=4, fmt="%.2f", fontsize=9.5)
ax.bar_label(rects_avg, padding=4, fmt="%.2f", fontsize=9.5, fontweight="bold")
ax.bar_label(rects_max, padding=4, fmt="%.2f", fontsize=9.5)

plt.tight_layout()
plt.savefig("working_hours_power_factor_summary_comparison.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Enforce clean default layout theme
plt.style.use('default')

# =====================================================================
# 1. APPLY WORKING HOURS & DAY FILTERS FIRST
# =====================================================================
working_df = abmf_hoffice_24_30_df.reset_index().copy()
working_df['timestamp'] = pd.to_datetime(working_df['timestamp'])
working_df['hour'] = working_df['timestamp'].dt.hour
working_df['day_name'] = working_df['timestamp'].dt.day_name()

# Apply strict filters: 8 AM to 5 PM inclusive, on Mon, Tue, Fri
time_mask = (working_df['hour'] >= 8) & (working_df['hour'] <= 17)
day_mask = working_df['day_name'].isin(['Monday', 'Tuesday', 'Friday'])
time_filtered_df = working_df[time_mask & day_mask]


# =====================================================================
# 2. CLEAN DATA & EXTRACT METRICS (FROM FILTERED WINDOW)
# =====================================================================
# Clean data: Filter out impossible sensor anomalies
valid_data = time_filtered_df[
    (time_filtered_df["apparent_power_overall_total"] > 0)
    & (
        time_filtered_df["load"]
        <= time_filtered_df["apparent_power_overall_total"]
    )
].copy()

# Calculate the instantaneous power factor per minute
valid_data["pf"] = (
    valid_data["load"] / valid_data["apparent_power_overall_total"]
)

# 3. Setup bins focusing on the typical operational window (0.00 to 1.00)
bins = np.arange(0, 1.01, 0.02)
centers = (bins[:-1] + bins[1:]) / 2

# Extract data splits
grid_pf = valid_data[valid_data["active_assets"] == "Grid"]["pf"].dropna()
gen_pf = (
    valid_data[valid_data["active_assets"] == "Generator 1"]["pf"].dropna()
)

# Compute distributions
grid_counts, _ = np.histogram(grid_pf, bins=bins)
gen_counts, _ = np.histogram(gen_pf, bins=bins)

# Convert minute-log counts to hours
grid_hours = grid_counts / 60.0
gen_hours = gen_counts / 60.0


# =====================================================================
# 4. CREATE SIDE-BY-SIDE PLOTS SHARING THE Y-AXIS (HOURS)
# =====================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

# Left Panel: Grid Power Factor Distribution
ax1.bar(
    centers,
    grid_hours,
    width=0.016,
    color="#1f77b4",
    edgecolor="black",
    alpha=0.85,
)
ax1.set_title("Grid Power Factor Stability", fontsize=12)
ax1.set_xlabel("Power Factor (kW / kVA)", fontsize=11)
ax1.set_ylabel("Operation Time (Hours)", fontsize=11)
ax1.set_xlim(0, 1.01)
ax1.grid(False)

# Remove the top and right borders (spines)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# Right Panel: Generator Power Factor Distribution
ax2.bar(
    centers,
    gen_hours,
    width=0.016,
    color="#ff7f0e",
    edgecolor="black",
    alpha=0.85,
)
ax2.set_title(
    "Generator 1 Power Factor Stability", fontsize=12
)
ax2.set_xlabel("Power Factor (kW / kVA)", fontsize=11)
ax2.set_xlim(0, 1.01)
ax2.grid(False)

# Remove the top and right borders (spines)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# Main Title updated to reflect specific working hours timeline
fig.suptitle(
    "Working Hours Power Factor Distribution Profile\n(Mon, Tue, Fri Only | 8:00 AM - 5:00 PM)",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)

plt.tight_layout()
plt.savefig("working_hours_power_factor_histograms.png", bbox_inches="tight")
plt.show()

In [ ]:
df.columns

In [ ]:
# --- CONFIGURATION & CONSTANTS ---
GENSET_CAPACITY_KW = 64  # Change this to your generator's actual size in kW
GENSET_EFF_LOW = 40  # Lower boundary of high-efficiency zone (%)
GENSET_EFF_HIGH = 80  # Upper boundary of high-efficiency zone (%)

# Colors matching your clean aesthetic
C_EFF = "#2ca02c"  # Green for the sweet-spot efficiency zone
C_NEUTRAL = "#b0bec5"  # Muted slate gray for under/over-utilized zones

# 1. Filter for Generator 1 data and drop missing records
gen_df = df[df["active_assets"] == "Generator 1"].dropna(subset=["load"])

# 2. Convert absolute kW load into a percentage of total capacity
# (If your 'load' column is already a 0-100% value, you can skip this calculation)
gen_load_pct = (gen_df["load"] / GENSET_CAPACITY_KW) * 100

# 3. Calculate the histogram distributions (0% to 100% in buckets of 10%)
bins = np.arange(0, 101, 10)
counts, edges = np.histogram(gen_load_pct, bins=bins)
centers = (edges[:-1] + edges[1:]) / 2

# 4. CONVERT MINUTES TO HOURS:
# Since data is logged every minute, divide the raw row counts by 60
counts_hours = counts / 60.0

# 5. Dynamically assign colors based on the efficiency sweet-spot
colors = [
    C_EFF if GENSET_EFF_LOW <= c < GENSET_EFF_HIGH else C_NEUTRAL
    for c in centers
]

# 6. Generate the sleek, low-height utilization plot
fig, ax = plt.subplots(figsize=(8, 3.5))

# Plot utilization bars
bars = ax.bar(centers, counts_hours, width=8, color=colors, edgecolor="black")

# Highlight the optimal efficiency window in the background
ax.axvspan(GENSET_EFF_LOW, GENSET_EFF_HIGH, color=C_EFF, alpha=0.06)

# Labels, titles, and layout formatting
ax.set_title("Generator Load Utilization Profile", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Generator Load (%)", fontsize=11, labelpad=8)
ax.set_ylabel("Operation Time (Hours)", fontsize=11, labelpad=8)

# Clean rendering adjustments
ax.set_xlim(0, 100)
ax.set_xticks(bins)

# --- ADDED CODE TO REMOVE BOX LINES ---
# Remove the top and right borders (spines)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# --------------------------------------

ax.grid(False)  # Completely removes background gridlines per preference

# Optional: Add text label inside the efficiency zone
ax.text(
    (GENSET_EFF_LOW + GENSET_EFF_HIGH) / 2,
    max(counts_hours) * 0.9,
    "Optimal Efficiency Zone",
    color=C_EFF,
    fontsize=10,
    fontweight="bold",
    ha="center",
)

plt.tight_layout()
#plt.savefig("generator_load_utilization.png", dpi=300)